# ATLAS tutorial: download CERES satellite solar radiation data

This notebook downloads monthly CERES satellite data from NASA Earthdata using `earthaccess`. Source: https://asdc.larc.nasa.gov/project/CERES/CER_SYN1deg-Month_Terra-Aqua-NOAA20_Edition4B

It is designed as a step-by-step tutorial for users who are not Python experts. In normal use, users only need to edit the **Input parameters** section and then run the notebook from top to bottom.

## What this notebook does

1. Imports the required Python packages.
2. Defines the input parameters: dataset, variable, dates and output folder.
3. Defines helper functions used for searching and downloading CERES files.
4. Runs the download.

## Important note about the spatial domain

The CERES download used here is **global**. For this reason, no country bounding box is required. The same notebook can be used by any country without changing a spatial parameter.

## Before running

You need a NASA Earthdata account. The first time you run `earthaccess.login(persist=True)`, you may be asked to enter your Earthdata username and password. The login can then be saved locally for future runs.

If `earthaccess` is not installed, install it with:

```bash
pip install earthaccess
```


## Step 1. Import required packages

Run this cell first. These packages are used to search and download the CERES data and to manage local folders.


In [1]:
from pathlib import Path
from datetime import datetime

import earthaccess


/home/alessandrom/anaconda3/envs/download_conda/lib/python3.1/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Step 2. Input parameters

Edit only this section for normal use.

### Dataset selection

- `CONCEPT_ID`: NASA Earthdata collection identifier for the CERES SYN1deg monthly product.
- `DATASET_KEYWORD`: keyword used only for optional dataset checks.
- `VARIABLE`: short name used in the ATLAS workflow. Here it is set to `"ghi"`.

### Dates

- `START_DATE`: first date to download. Here it starts from **1 January 2000**.
- `END_DATE`: last date to download. Here it ends in **March 2026**.

### Output folder

- `OUTPUT_DIR`: folder where the downloaded CERES files will be saved.
- The default structure is `../data/ceres_downloads/{variable}/`, therefore with `VARIABLE = "ghi"` the output folder is `../data/ceres_downloads/ghi/`.

### Spatial domain

No bounding box is used. The CERES files are downloaded globally and can later be subset or processed for any country.


In [2]:
# =========================
# DATASET SELECTION
# =========================

# NASA Earthdata collection ID for CERES SYN1deg monthly Terra-Aqua-NOAA20 Edition 4B
CONCEPT_ID = "C3880454295-LARC_CLOUD"

# Optional keyword used to check the dataset collection
DATASET_KEYWORD = "CER_SYN1deg-Month_Terra-Aqua-NOAA20_Edition4B"

# Short variable name used by the ATLAS workflow
VARIABLE = "ghi"


# =========================
# DATE RANGE
# =========================

# Download period requested for the tutorial. Saving one file per month
START_DATE = "2000-01-01"
END_DATE = "2026-03-31"


# =========================
# OUTPUT FOLDER
# =========================

# Output path requested for the ATLAS workflow
OUTPUT_DIR = Path("../data/ceres_downloads") / VARIABLE


# =========================
# DOWNLOAD OPTIONS
# =========================

# Maximum number of granules returned by the search.
# The period 2000-01 to 2026-03 contains about 315 monthly files, so 1000 is sufficient.
COUNT = 1000

# Set to False to avoid downloading files again when they already exist locally.
OVERWRITE_EXISTING_FILES = False


## Step 3. Optional dataset check

Run this cell if you want to verify that the CERES collection can be found through NASA Earthdata.

This step is not strictly required for the final download, but it is useful when testing the notebook for the first time.


In [3]:
earthaccess.login(persist=True)

collections = earthaccess.search_datasets(keyword=DATASET_KEYWORD)

print("Collections found:", len(collections))
for collection in collections[:3]:
    print(collection)


Collections found: 1
{
  "meta": {
    "revision-id": 7,
    "deleted": false,
    "format": "application/vnd.nasa.cmr.umm+json",
    "provider-id": "LARC_CLOUD",
    "has-combine": false,
    "user-id": "etshoema",
    "has-formats": false,
    "s3-links": [
      "s3://asdc-prod-protected/CERES/CER_SYN1deg-Month_Terra-Aqua-NOAA20_Edition4B"
    ],
    "has-spatial-subsetting": false,
    "native-id": "CER_SYN1deg-Month_Terra-Aqua-NOAA20_Edition4B",
    "has-transforms": false,
    "has-variables": false,
    "concept-id": "C3880454295-LARC_CLOUD",
    "revision-date": "2026-05-12T19:05:39.798Z",
    "has-temporal-subsetting": false,
    "concept-type": "collection"
  },
  "umm": {
    "AncillaryKeywords": [
      "Atmosphere Incoming Shortwave (SW) Entropy",
      "Atmosphere Outgoing Longwave (LW) Entropy",
      "Clear-Sky Fluxes",
      "Cloud Effective Height",
      "Cloud Effective Pressure",
      "Cloud Effective Temperature",
      "Cloud Infrared Emissivity",
      "Cloud P

## Step 4. Functions

Run this cell without editing it.

The functions below:

- validate the main user inputs;
- create the output folder;
- log in to NASA Earthdata;
- search CERES monthly granules for the selected date range;
- download the files to the selected output folder.


In [4]:
def validate_inputs(concept_id, variable, start_date, end_date, output_dir, count):
    """
    Validate the main notebook inputs before starting the download.
    """
    if not concept_id:
        raise ValueError("CONCEPT_ID cannot be empty.")

    if not variable:
        raise ValueError("VARIABLE cannot be empty.")

    try:
        start = datetime.strptime(start_date, "%Y-%m-%d")
        end = datetime.strptime(end_date, "%Y-%m-%d")
    except ValueError as exc:
        raise ValueError("START_DATE and END_DATE must use the format YYYY-MM-DD.") from exc

    if start > end:
        raise ValueError("START_DATE must be earlier than or equal to END_DATE.")

    if count <= 0:
        raise ValueError("COUNT must be a positive integer.")

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    return output_dir


def month_string(date_text):
    """
    Convert a date string in YYYY-MM-DD format to YYYY-MM format.
    NASA Earthdata temporal searches for this collection are performed by month.
    """
    return datetime.strptime(date_text, "%Y-%m-%d").strftime("%Y-%m")


def search_ceres_monthly_granules(concept_id, start_date, end_date, count):
    """
    Search CERES monthly granules for the selected period.

    The search is global. No bounding box is passed to earthaccess.search_data().
    """
    temporal_range = (month_string(start_date), month_string(end_date))

    granules = earthaccess.search_data(
        concept_id=concept_id,
        temporal=temporal_range,
        count=count,
    )

    if not granules:
        raise RuntimeError(
            f"No CERES granules found for concept_id={concept_id} "
            f"between {temporal_range[0]} and {temporal_range[1]}."
        )

    return granules


def download_ceres_monthly_global(
    concept_id,
    variable,
    start_date,
    end_date,
    output_dir,
    count=1000,
    overwrite=False,
):
    """
    Download global monthly CERES files to the selected output folder.
    """
    output_dir = validate_inputs(
        concept_id=concept_id,
        variable=variable,
        start_date=start_date,
        end_date=end_date,
        output_dir=output_dir,
        count=count,
    )

    earthaccess.login(persist=True)

    granules = search_ceres_monthly_granules(
        concept_id=concept_id,
        start_date=start_date,
        end_date=end_date,
        count=count,
    )

    print(f"Granules found: {len(granules)}")
    print(f"Output folder: {output_dir.resolve()}")

    files = earthaccess.download(
        granules,
        local_path=str(output_dir),
        threads=4,
        pqdm_kwargs={"disable": False},
    )

    if not overwrite:
        print("Note: earthaccess may skip files that already exist locally.")

    return files


## Step 5. Run the download

Run this cell after checking the input parameters above.

The download is global and saves the files to:

```text
../data/ceres_downloads/ghi/
```


In [5]:
downloaded_files = download_ceres_monthly_global(
    concept_id=CONCEPT_ID,
    variable=VARIABLE,
    start_date=START_DATE,
    end_date=END_DATE,
    output_dir=OUTPUT_DIR,
    count=COUNT,
    overwrite=OVERWRITE_EXISTING_FILES,
)

print("Downloaded files:", len(downloaded_files))
for file_path in downloaded_files[:10]:
    print(file_path)

if len(downloaded_files) > 10:
    print(f"... and {len(downloaded_files) - 10} more files")


/home/alessandrom/anaconda3/envs/download_conda/lib/python3.13/site-packages/earthaccess/results.py:343: FutureWarning: As of version 1.0, `DataGranule.size` will be accessed as an attribute; e.g. use `DataCollection.size` **not** `DataCollection.size()`
  self["size"] = self.size()
/home/alessandrom/anaconda3/envs/download_conda/lib/python3.13/site-packages/earthaccess/store.py:832: FutureWarning: As of version 1.0, `DataGranule.size` will be accessed as an attribute; e.g. use `DataCollection.size` **not** `DataCollection.size()`
  total_size = round(sum(granule.size() for granule in granules) / 1024, 2)


Granules found: 13
Output folder: /home/python/jupyters/WMO_ATLAS/Notebooks_for_Deliverable_3/data/ceres_downloads/ghi


QUEUEING TASKS | : 100%|██████████| 13/13 [00:00<00:00, 1721.20it/s]
PROCESSING TASKS | : 100%|██████████| 13/13 [00:17<00:00,  1.38s/it]
COLLECTING RESULTS | : 100%|██████████| 13/13 [00:00<00:00, 53300.05it/s]

Note: earthaccess may skip files that already exist locally.
Downloaded files: 13
../data/ceres_downloads/ghi/CER_SYN1deg-Month_Terra-Aqua-NOAA20_Edition4B_401412.200003
../data/ceres_downloads/ghi/CER_SYN1deg-Month_Terra-Aqua-NOAA20_Edition4B_401412.200004
../data/ceres_downloads/ghi/CER_SYN1deg-Month_Terra-Aqua-NOAA20_Edition4B_401412.200005
../data/ceres_downloads/ghi/CER_SYN1deg-Month_Terra-Aqua-NOAA20_Edition4B_401412.200006
../data/ceres_downloads/ghi/CER_SYN1deg-Month_Terra-Aqua-NOAA20_Edition4B_401412.200007
../data/ceres_downloads/ghi/CER_SYN1deg-Month_Terra-Aqua-NOAA20_Edition4B_401412.200008
../data/ceres_downloads/ghi/CER_SYN1deg-Month_Terra-Aqua-NOAA20_Edition4B_401412.200009
../data/ceres_downloads/ghi/CER_SYN1deg-Month_Terra-Aqua-NOAA20_Edition4B_401412.200010
../data/ceres_downloads/ghi/CER_SYN1deg-Month_Terra-Aqua-NOAA20_Edition4B_401412.200011
../data/ceres_downloads/ghi/CER_SYN1deg-Month_Terra-Aqua-NOAA20_Edition4B_401412.200012
... and 3 more files


## Notes for adapting this notebook

This notebook is already valid for every country because the CERES download is global.

For a new country, users do **not** need to change a bounding box. They only need to:

1. keep the same global CERES download;
2. run the notebook from top to bottom;
3. use the downloaded files later in the country-specific processing workflow.

The default period is from **1 January 2000** to **31 March 2026**.
